In [2]:
import os
os.listdir()

['.ipynb_checkpoints', 'EDA_Process.ipynb']

In [4]:
!pip install seaborn

In [6]:
!pip install WordCloud

In [7]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')            
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

In [8]:
POS  = '#1D9E75'   # green = positive
NEU  = '#EF9F27'   # amber = neutral
NEG  = '#D85A30'   # red  = negative

In [9]:
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
})

In [11]:
df = pd.read_csv('cleaned_reviews.csv')
print(f"Loaded {len(df)} clean reviews")
print(df['sentiment'].value_counts())

Loaded 1162 clean reviews
sentiment
positive    964
neutral     124
negative     74
Name: count, dtype: int64


# CHART 1 — Sentiment Distribution 

In [12]:
counts = df['sentiment'].value_counts()
labels = counts.index.tolist()
sizes  = counts.values.tolist()
colors = [POS if l=='positive' else NEU if l=='neutral' else NEG for l in labels]

fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    sizes, labels=None, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    pctdistance=0.75
)
for at in autotexts:
    at.set(fontsize=12, fontweight='bold', color='white')

ax.text(0, 0, f'{sum(sizes)}\nreviews', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#2C2C2A')

legend_patches = [mpatches.Patch(color=c, label=f'{l.title()}  ({s})')
                  for c, l, s in zip(colors, labels, sizes)]
ax.legend(handles=legend_patches, loc='lower center',
          bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=11)

ax.set_title('Sentiment Distribution', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('chart1_sentiment.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart 1 saved: chart1_sentiment.png")


Chart 1 saved: chart1_sentiment.png


# CHART 2 — Star Rating Bar Chart

In [13]:
rating_counts = df['rating'].value_counts().sort_index()
bar_colors = [NEG, NEG, NEU, POS, POS]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(rating_counts.index, rating_counts.values,
              color=bar_colors, width=0.6, edgecolor='white', linewidth=1.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 8,
            str(int(bar.get_height())),
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#2C2C2A')

patches = [mpatches.Patch(color=NEG, label='Negative (1-2 stars)'),
           mpatches.Patch(color=NEU, label='Neutral (3 stars)'),
           mpatches.Patch(color=POS, label='Positive (4-5 stars)')]
ax.legend(handles=patches, frameon=False, fontsize=9, loc='upper left')
ax.set_xlabel('Star Rating', fontsize=12)
ax.set_title('How Many Reviews Per Star Rating?', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1 star', '2 stars', '3 stars', '4 stars', '5 stars'])
ax.set_ylim(0, rating_counts.max() * 1.18)
ax.yaxis.set_visible(False)

plt.tight_layout()
plt.savefig('chart2_ratings.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart 2 saved: chart2_ratings.png")

Chart 2 saved: chart2_ratings.png


# CHART 3 — Word Cloud: Positive Reviews

In [14]:
pos_text = ' '.join(df[df['sentiment']=='positive']['clean_text'].dropna())
wc = WordCloud(width=700, height=360, background_color='white',
               colormap='Spectral', max_words=80, collocations=False).generate(pos_text)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Common Words in POSITIVE Reviews', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('chart3_wordcloud_positive.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart 3 saved: chart3_wordcloud_positive.png")



Chart 3 saved: chart3_wordcloud_positive.png


# CHART 4 — Word Cloud: Negative Reviews

In [15]:
neg_text = ' '.join(df[df['sentiment']=='negative']['clean_text'].dropna())
wc2 = WordCloud(width=700, height=360, background_color='white',
                colormap='OrRd', max_words=80, collocations=False).generate(neg_text)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.imshow(wc2, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Common Words in NEGATIVE Reviews', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('chart4_wordcloud_negative.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart 4 saved: chart4_wordcloud_negative.png")

Chart 4 saved: chart4_wordcloud_negative.png


# CHART 5 — Top 10 Words Side by Side

In [16]:
def top_words(sentiment, n=10):
    text = ' '.join(df[df['sentiment']==sentiment]['clean_text'].dropna())
    return Counter(text.split()).most_common(n)

pos_words = top_words('positive')
neg_words = top_words('negative')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

pw, pc = zip(*pos_words)
ax1.barh(list(pw)[::-1], list(pc)[::-1], color=POS, edgecolor='white')
ax1.set_title('Top 10 Words — Positive', fontsize=12, fontweight='bold')
ax1.set_xlabel('Count')
for i, v in enumerate(list(pc)[::-1]):
    ax1.text(v+2, i, str(v), va='center', fontsize=9)

nw, nc = zip(*neg_words)
ax2.barh(list(nw)[::-1], list(nc)[::-1], color=NEG, edgecolor='white')
ax2.set_title('Top 10 Words — Negative', fontsize=12, fontweight='bold')
ax2.set_xlabel('Count')
for i, v in enumerate(list(nc)[::-1]):
    ax2.text(v+0.5, i, str(v), va='center', fontsize=9)

plt.suptitle('What words appear most in each sentiment?', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chart5_topwords.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart 5 saved: chart5_topwords.png")

Chart 5 saved: chart5_topwords.png


# CHART 6 — Review Length Distribution

In [17]:
df['review_length'] = df['clean_text'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(7, 4))
for sentiment, color in [('positive', POS), ('neutral', NEU), ('negative', NEG)]:
    subset = df[df['sentiment']==sentiment]['review_length']
    ax.hist(subset, bins=30, alpha=0.6, color=color,
            label=f'{sentiment.title()} (avg {int(subset.mean())} words)',
            edgecolor='white')

ax.set_xlabel('Number of words in review (after cleaning)', fontsize=11)
ax.set_ylabel('Number of reviews', fontsize=11)
ax.set_title('How Long Are Reviews?', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, frameon=False)
plt.tight_layout()
plt.savefig('chart6_length.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart 6 saved: chart6_length.png")

Chart 6 saved: chart6_length.png


# CHART 7 — Top 5 Most Reviewed Products

In [18]:
top_products = df['product'].value_counts().head(5)
colors7 = [POS, '#3B8BD4', NEU, NEG, '#888780']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(top_products.index[::-1], top_products.values[::-1],
               color=colors7, edgecolor='white')
for bar in bars:
    ax.text(bar.get_width()+4, bar.get_y()+bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontsize=10)

ax.set_xlabel('Number of Reviews', fontsize=11)
ax.set_title('Top 5 Most Reviewed Products', fontsize=14, fontweight='bold')
ax.xaxis.set_visible(False)
plt.tight_layout()
plt.savefig('chart7_products.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart 7 saved: chart7_products.png")

Chart 7 saved: chart7_products.png


In [20]:
print("\n" + "="*70)
print("KEY INSIGHTS OF REPORT")

print(f"Total reviews analysed : {len(df)}")
print(f"Positive               : {len(df[df.sentiment=='positive'])} ({len(df[df.sentiment=='positive'])*100//len(df)}%)")
print(f"Neutral                : {len(df[df.sentiment=='neutral'])} ({len(df[df.sentiment=='neutral'])*100//len(df)}%)")
print(f"Negative               : {len(df[df.sentiment=='negative'])} ({len(df[df.sentiment=='negative'])*100//len(df)}%)")
print(f"Avg review length      : {int(df['review_length'].mean())} words")
print(f"Avg length (positive)  : {int(df[df.sentiment=='positive']['review_length'].mean())} words")
print(f"Avg length (negative)  : {int(df[df.sentiment=='negative']['review_length'].mean())} words")
print(f"Most reviewed product  : {df['product'].value_counts().index[0]}")
print("="*70)


KEY INSIGHTS OF REPORT
Total reviews analysed : 1162
Positive               : 964 (82%)
Neutral                : 124 (10%)
Negative               : 74 (6%)
Avg review length      : 69 words
Avg length (positive)  : 66 words
Avg length (negative)  : 50 words
Most reviewed product  : Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker
